# StreamVault Exploratory Data Profiling

Investigation, not production code. It reads the source data as text so nothing is converted or
hidden prematurely, profiles every column, checks every claim in the brief against reality,
and reveals the exact rows behind the numbers.

**Architecture reference:** Modeled on Lecture 2 (`profile_day.ipynb`).

**To use it:** *Run All* cells in order.

**Before you commit:** Clear all outputs if you prefer clean version control.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

DATA = Path("../data") if Path.cwd().name == "profiling" else Path("data")
RAW = DATA / "raw"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)

## 1. The candidate list, exactly as received

In [ ]:
def load():
    """Read candidate lines directly from id_list.csv without dropping blank rows."""
    with open(DATA / "id_list.csv", "r", encoding="utf-8") as f:
        lines = f.read().splitlines()
    
    header = lines[0]
    data_lines = lines[1:]
    
    raw = pd.DataFrame({
        "row_number": range(1, len(data_lines) + 1),
        "raw_id": data_lines
    })
    return raw

raw = load()
print(f"Read {len(raw)} candidate lines from id_list.csv (expected 127)")
raw.head(15)

In [ ]:
# Inspect a sample raw JSON response from data/raw/ (if fetched)
sample_file = RAW / "155_details.json"
if sample_file.exists():
    with open(sample_file, "r", encoding="utf-8") as f:
        sample_data = json.load(f)
    print(f"Title: {sample_data.get('title')} | ID: {sample_data.get('id')}")
    print(f"Collection: {sample_data.get('belongs_to_collection')}")
    print(f"Genres: {sample_data.get('genres')}")
else:
    print("Raw JSON files not yet present. Run: python -m pipeline.extract")

## 2. Every column: blanks, distinct values, commonest values

In [ ]:
def columns(df: pd.DataFrame) -> pd.DataFrame:
    """One row per column: how many blanks, how many distinct values, the commonest values."""
    results = []
    for c in df.columns:
        s = df[c].astype(str).str.strip()
        blanks = int(s.isin(["", "nan", "None"]).sum())
        valid = df.loc[~s.isin(["", "nan", "None"]), c].astype(str)
        distinct = int(valid.nunique())
        commonest = ", ".join(valid.value_counts().index[:3])
        results.append({
            "column": c,
            "blank": blanks,
            "distinct": distinct,
            "commonest (not blank)": commonest
        })
    return pd.DataFrame(results).set_index("column")

print("Candidate ID Column Profile:")
columns(raw)

In [ ]:
# Profile the flattened movies dataset (1 row per film grain)
movies_file = DATA / "movies.csv"
if movies_file.exists():
    df_movies = pd.read_csv(movies_file)
    print(f"Movies Table Profile: {len(df_movies)} rows, unique tmdb_id: {df_movies['tmdb_id'].is_unique}")
    display(columns(df_movies))
else:
    print("movies.csv not yet generated. Run: python -m pipeline.transform")

## 3. The spec's claims, checked

One line per investigation question, answered with exact numbers:
- Does the list contain 127 film identifiers?
- How many duplicates and invalid formats are present?
- How many IDs return HTTP 404 from TMDb?
- What is the distribution of franchise bundles vs. standalones?
- How many titles fall before vs. after 2000 (Yaw in Finance's concern)?

In [ ]:
def checks(raw: pd.DataFrame):
    """Answer the brief's questions with exact numbers."""
    s = raw["raw_id"].str.strip()
    
    # Validation tests
    is_empty = s.eq("")
    numeric_val = pd.to_numeric(s, errors="coerce")
    is_invalid_str = s.ne("") & numeric_val.isna()
    is_negative = numeric_val.le(0)
    is_valid_int = numeric_val.gt(0)
    
    valid_ids = numeric_val[is_valid_int].astype(int)
    duplicates = valid_ids.duplicated()
    unique_candidates = valid_ids.drop_duplicates()
    
    # Output summary dict
    report = {
        "Total lines in id_list.csv (excluding header)": len(raw),
        "Brief stated count": 127,
        "Blank / empty rows in file": int(is_empty.sum()),
        "Non-numeric string IDs ('tt0468569', 'N/A')": int(is_invalid_str.sum()),
        "Non-positive integer IDs ('-1')": int(is_negative.sum()),
        "Total syntactically invalid rows": int(is_empty.sum() + is_invalid_str.sum() + is_negative.sum()),
        "Duplicate candidate IDs": int(duplicates.sum()),
        "Unique valid IDs to extract": len(unique_candidates),
    }
    
    # Add fetch log results if available
    log_file = DATA / "fetch_log.csv"
    if log_file.exists():
        log_df = pd.read_csv(log_file)
        disp = log_df["disposition"].value_counts().to_dict()
        report["Fetch log accounted total"] = len(log_df)
        report["Successfully fetched (200 OK)"] = disp.get("FETCHED", 0)
        report["Fictional IDs (HTTP 404 Not Found)"] = disp.get("FETCH_FAILED", 0)
        
    # Add movies dataset results if available
    if (DATA / "movies.csv").exists():
        m = pd.read_csv(DATA / "movies.csv")
        report["Franchise titles in catalog"] = int(m["is_franchise"].sum())
        report["Standalone titles in catalog"] = int((~m["is_franchise"]).sum())
        report["Pre-2000 titles (Finance concern)"] = int((m["release_year"] < 2000).sum())
        report["2000-onward titles (Finance concern)"] = int((m["release_year"] >= 2000).sum())
        report["Titles with zero budget"] = int((m["budget"] == 0).sum())
        report["Titles with zero revenue"] = int((m["revenue"] == 0).sum())
        
    return report

pd.DataFrame({"StreamVault Candidate Audit": checks(raw)})

## 4. The rows behind the numbers

Counts tell you *how many*. Rows tell you *why*.

In [ ]:
# Repeated candidate IDs, side by side
s = raw["raw_id"].str.strip()
numeric_ids = pd.to_numeric(s, errors="coerce")
dup_mask = numeric_ids.duplicated(keep=False) & numeric_ids.notna()

print("Duplicate rows in id_list.csv (side by side):")
raw[dup_mask].sort_values("raw_id")

In [ ]:
# Syntactically invalid candidate rows
invalid_mask = s.eq("") | numeric_ids.isna() | numeric_ids.le(0)

print("Syntactically invalid rows:")
raw[invalid_mask]

In [ ]:
# Non-existent candidate IDs (HTTP 404 Not Found from TMDb)
log_file = DATA / "fetch_log.csv"
if log_file.exists():
    log_df = pd.read_csv(log_file)
    failed_404s = log_df[log_df["disposition"] == "FETCH_FAILED"]
    print("Fictional / non-existent IDs in candidate list (HTTP 404):")
    display(failed_404s[["row", "raw", "reason"]])
else:
    print("fetch_log.csv not yet generated.")

In [ ]:
# Top Franchise Collections in catalog by total box office
if (DATA / "movies.csv").exists():
    m = pd.read_csv(DATA / "movies.csv")
    collections = (
        m[m["is_franchise"]]
        .groupby(["collection_id", "collection_name"])
        .agg(
            titles=("tmdb_id", "count"),
            total_revenue=("revenue", "sum"),
            mean_revenue=("revenue", "mean"),
            mean_vote=("vote_average", "mean")
        )
        .reset_index()
        .sort_values(by="total_revenue", ascending=False)
    )
    
    top_collections = collections.head(8).copy()
    top_collections["total_revenue"] = top_collections["total_revenue"].apply(lambda x: f"${x:,.0f}")
    top_collections["mean_revenue"] = top_collections["mean_revenue"].apply(lambda x: f"${x:,.0f}")
    top_collections["mean_vote"] = top_collections["mean_vote"].apply(lambda x: f"{x:.2f}")
    
    print("Top Franchise Collections in candidate catalog:")
    display(top_collections)
else:
    print("movies.csv not yet generated.")

In [ ]:
# Dual-Window Comparison: All Years vs. 2000-Onward (Yaw in Finance)
if (DATA / "movies.csv").exists():
    m = pd.read_csv(DATA / "movies.csv")
    m["roi"] = (m["revenue"] - m["budget"]) / m["budget"]
    
    def window_kpis(df, window_label):
        agg = df.groupby("is_franchise").agg(
            count=("tmdb_id", "count"),
            mean_revenue=("revenue", "mean"),
            median_revenue=("revenue", "median"),
            mean_roi=("roi", "mean"),
            median_roi=("roi", "median"),
            mean_vote=("vote_average", "mean")
        ).reset_index()
        agg["type"] = np.where(agg["is_franchise"], "Franchise Bundle", "Standalone Hit")
        agg["window"] = window_label
        return agg[["window", "type", "count", "mean_revenue", "median_revenue", "mean_roi", "median_roi", "mean_vote"]]
    
    kpi_all = window_kpis(m, "All Years")
    kpi_post_2000 = window_kpis(m[m["release_year"] >= 2000], "2000-Onward")
    comp = pd.concat([kpi_all, kpi_post_2000], ignore_index=True)
    
    display_comp = comp.copy()
    display_comp["mean_revenue"] = display_comp["mean_revenue"].apply(lambda x: f"${x:,.0f}")
    display_comp["median_revenue"] = display_comp["median_revenue"].apply(lambda x: f"${x:,.0f}")
    display_comp["mean_roi"] = display_comp["mean_roi"].apply(lambda x: f"{x:.2f}x")
    display_comp["median_roi"] = display_comp["median_roi"].apply(lambda x: f"{x:.2f}x")
    display_comp["mean_vote"] = display_comp["mean_vote"].apply(lambda x: f"{x:.2f}")
    
    print("Two-Window Comparison of Franchise Bundles vs. Standalone Hits:")
    display(display_comp)
else:
    print("movies.csv not yet generated.")